In [41]:
import os
import sys

import logging

%pwd
os.chdir("../")

# Add the absolute path to src/ so Python can find automatch
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.append(src_path)
    
%pwd


from src.onBench.utils.EDA import *


logging.getLogger("snowflake").setLevel(logging.WARNING)
logging.getLogger("snowflake.connector").setLevel(logging.WARNING)
logging.getLogger("snowflake.snowpark").setLevel(logging.WARNING)





In [42]:
from src.onBench.utils.snowflake_utils import get_snowpark_session
session = get_snowpark_session()

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/e2ba81b8-03fe-407c-96a1-f4bc0f512e7d/saml2?SAMLRequest=nZJNb%2BIwEIb%2FSuQ9J3HMtwVUUARFKgsqoYt6M86EeuvYrO00wK9fE4rUPbSHlXKInGfGT%2Bad%2Ft2xkME7GCu0GqAkwigAxXUm1H6ANuk07KLAOqYyJrWCATqBRXfDvmWFPNBR6V7VE%2FwpwbrAN1KW1h8GqDSKamaFpYoVYKnjdD1aPFISYXow2mmuJfpU8n0FsxaM84a3kswKr%2Ffq3IHGcVVVUdWItNnHBGMc417sqQvy48Yf%2FT99wScxbl54T3h89eE2Fuo6gu%2B0dlfI0oc0XYWr5TpFweimeq%2BVLQswazDvgsPm6fEqYL3BeN1rYtKNKj%2B3EEqjDxCxc2kgskpXuWRvwHVxKJ3vHvm3OIcslnov%2FADmkwE6vIksXbQyfpodN9uML%2FPt78Yznm2nM7kx4%2F3D%2BaVxXpglU%2FvOr5blKHi%2BJUwuCc%2BtLWGuLrk6f4RJO8TtkJAUd2nDPzhqE%2FKCgon3E4q5uvImX3tEheBGW507raRQUFsC2bFusuuGuJFD2MQdHvbaLAnz5o7jvJUQ6GTxJW2CrhtEaxEz%2FN%2B59OPPXT6W8qfPaT5ZaSn4KZhqUzD3dYxJlNQnIgvzGqVQMCFHWWbAWh%2BnlLq6N8Cc331n

In [43]:
df = session.sql(f"""select *
                     from it_discovery.consumer_int_model.MPG_IT_ONBENCH_CANDIDATE
                     where status = 'No mission'
                 """).to_pandas()


[2026-06-22 10:38:36,972: ERROR: ocsp_snowflake: WARNING!!! Using fail-open to connect. Driver is connecting to an HTTPS endpoint without OCSP based Certificate Revocation checking as it could not obtain a valid OCSP Response to use from the CA OCSP responder. Details: 
 {'driver': 'PythonConnector', 'version': '3.7.0', 'eventType': 'RevocationCheckFailure', 'eventSubType': 'OCSPResponseAttachedCertInvalid', 'sfcPeerHost': 'iojcjrsfcb1stg.blob.core.windows.net', 'certId': 'MEowBwYFKw4DAhoEFDRxGqvIMaRG9VdxNofHFsCKo4HoBBS4LzOmfFFPfx8t1cgVQjiksg6PLwITQQAVVfKGPz0rumHUNAAAABVV8g==', 'ocspRequestBase64': <asn1crypto.ocsp.OCSPRequest 2800294196176 b'0R0P0N0L0J0\x07\x06\x05+\x0e\x03\x02\x1a\x04\x144q\x1a\xab\xc81\xa4F\xf5Wq6\x87\xc7\x16\xc0\x8a\xa3\x81\xe8\x04\x14\xb8/3\xa6|QO\x7f\x1f-\xd5\xc8\x15B8\xa4\xb2\x0e\x8f/\x02\x13A\x00\x15U\xf2\x86?=+\xbaa\xd44\x00\x00\x00\x15U\xf2'>, 'ocspResponderURL': 'http://oneocsp.microsoft.com/ocsp', 'errorMessage': '254009: Failed to verify the signature', 'i

In [44]:
df["CCNL_CODE"] = df["CCNL_CODE"].astype("Int64").astype("string")

df[df.select_dtypes(include=["object", "string"]).columns] = (
    df.select_dtypes(include=["object", "string"]).apply(lambda col: col.str.strip())
)


In [45]:
keywords = [

    #general
    "curriculum", "cv", "vitae", "esperienza", "lavoro", "occupazione", "posizione", "ruolo", "azienda",
    # ruoli
    "addetto", "operaio", "impiegato", "responsabile", "cassiere", "magazziniere",
    "analista", "developer", "autista", "segretaria", "tecnico", "consulente",

    # titoli di studio
    "diploma", "laurea", "master", "istituto", "università",

    # competenze
    "excel", "office", "inglese", "informatica", "patente",

    # pattern temporali
    #"2019", "2020", "2021", "2022", "2023", "2024", "dal", "al",

    # aziende / enti
    "srl", "spa", "cooperativa", "comune", "regione"
]

df["IS_VALID_CV"] = df["DESCRIPTION"].str.lower().apply(
    lambda x: any(k in x for k in keywords) if isinstance(x, str) else False
)



In [46]:
general_info(df)


Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   CANDIDATE_ID_BH   365 non-null    int32         
 1   DBJOBCANDNO       365 non-null    int32         
 2   JOB_SITE          365 non-null    object        
 3   JOB_FIRST         365 non-null    object        
 4   JOB_INTERMEDIATE  144 non-null    object        
 5   JOB_LAST          245 non-null    object        
 6   START_DATE        365 non-null    datetime64[ns]
 7   END_DATE          365 non-null    datetime64[ns]
 8   PARTTIME_PERC     41 non-null     float64       
 9   CCNL_LEVEL        365 non-null    object        
 10  LEGAL_CATEGORY    365 non-null    object        
 11  CCNL_CODE         359 non-null    string        
 12  CCNL_DESCRIPTION  359 non-null    object        
 13  STATUS            365 non-null    object        
 14  SKILLS            9

In [52]:
columns_to_check = ["EDUCATION", "CERTIFICATIONS", "IS_VALID_CV"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)


Unique values in 'EDUCATION':
['Bachelor' 'Diploma' 'Laurea' 'Laurea Breve' 'Non Specificato'
 'Scuola Media' 'Scuola Secondaria' 'Universita' None]
----------------------------------------
Unique values in 'CERTIFICATIONS':
[None]
----------------------------------------
Unique values in 'IS_VALID_CV':
[False  True]
----------------------------------------


In [53]:
columns_to_check = ["CCNL_LEVEL", "CCNL_CODE", "LEGAL_CATEGORY", "CCNL_DESCRIPTION"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)



Unique values in 'CCNL_LEVEL':
['1' '2' '3' '4' '4 Junior' '4 Senior' '4B' '5' '6' '6 Junior' '6 Senior'
 '6 Super (ex 7)' '6 junior' '7' '8' 'A' 'A1' 'AE1' 'AP SECONDA'
 'Area 3 Liv. 1' 'B1' 'B3' 'C' 'C Super' 'C/1' 'C1' 'C2' 'C3' 'D' 'D/1'
 'D/2' 'D1' 'D2' 'D3' 'E' 'E2' 'E3' 'E4' 'G' 'G1' 'H' 'I' 'III' 'Junior'
 'Op Centrato' 'Operat esperti' 'V']
----------------------------------------
Unique values in 'CCNL_CODE':
<StringArray>
[ '106', '1200', '1270',  '130', '1380', '1390', '1790', '1810', '1820',
 '1870', '1930', '1980',   '20',  '200', '2070',  '230', '2310', '2360',
  '240',  '250', '2560', '3040',  '330',  '370',   '38', '3980',   '46',
  '530',  '710',  '720',   '81', '9013', '9050', '9332', '9335', '9354',
 '9357', '9408', '9409', '9443', '9504', '9631', '9646', '9662', '9683',
 '9694', '9705', '9719', '9726', '9746', '9753', '9763', '9776', '9789',
 '9796', '9797',  '980', '9817', '9850',   <NA>]
Length: 60, dtype: string
----------------------------------------
Unique va

In [49]:
b1 = bar_plot(df, 'EDUCATION')#, label_range=30)
b2 = bar_plot(df, 'CERTIFICATIONS')#, label_range=30)
b3 = bar_plot(df, 'IS_VALID_CV')#, label_range=30)

(b1 + b2 + b3).cols(1)

:Layout
   .Bars.I   :Bars   [EDUCATION]   (COUNT)
   .Bars.II  :Bars   [CERTIFICATIONS]   (COUNT)
   .Bars.III :Bars   [IS_VALID_CV]   (COUNT)

In [50]:
b1 = bar_plot(df, 'CCNL_LEVEL')#, label_range=30)
b2 = bar_plot(df, 'CCNL_CODE')#, label_range=30)
b3 = bar_plot(df, 'LEGAL_CATEGORY')#, label_range=30)
b4 = bar_plot(df, 'CCNL_DESCRIPTION', rotation=25)

(b1 + b2 + b3 + b4).cols(1)

:Layout
   .Bars.I   :Bars   [CCNL_LEVEL]   (COUNT)
   .Bars.II  :Bars   [CCNL_CODE]   (COUNT)
   .Bars.III :Bars   [LEGAL_CATEGORY]   (COUNT)
   .Bars.IV  :Bars   [CCNL_DESCRIPTION]   (COUNT)

In [51]:
h1 = hist_plot(df, "PARTTIME_PERC", bins=20, xrange=[], yrange=[], height=400, width=800)

h1

:Histogram   [PARTTIME_PERC]   (Frequency)

# VACANCIES